# 04 — Training the Neural ODE Autoencoder

This notebook trains the Neural ODE Autoencoder on benign-only network traffic and visualizes the training process.

**Training setup:**
- Loss: MSE reconstruction error (benign windows only)
- Optimizer: Adam (lr=1e-3, weight_decay=1e-5)
- Scheduler: Cosine Annealing
- Early stopping: patience=5 on benign validation loss

In [1]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from IPython.display import clear_output

from src.model import NeuralODEAutoencoder
from src.dataset import get_dataloaders

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'MPS available:  {torch.backends.mps.is_available()}')

PyTorch 2.11.0
CUDA available: False
MPS available:  True


## 1. Setup

In [2]:
# Device selection — torchdiffeq has limited MPS support, so we use CPU by default.
# If you have CUDA, it will be used automatically.
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

# Reproducibility
SEED = config['preprocessing']['random_state']
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cpu


In [3]:
# Load data
batch_size = config['training']['batch_size']
loaders = get_dataloaders('../data/processed', batch_size)

for split, loader in loaders.items():
    ds = loader.dataset
    n_attack = int(ds.y.sum())
    n_benign = len(ds) - n_attack
    print(f'{split:5s}: {len(ds):,} windows  (benign={n_benign:,}, attack={n_attack:,})')

train: 69,918 windows  (benign=69,918, attack=0)
val  : 18,959 windows  (benign=14,982, attack=3,977)
test : 18,959 windows  (benign=14,982, attack=3,977)


In [4]:
# Create model
model = NeuralODEAutoencoder(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
print(f'Model size: ~{n_params * 4 / 1024 / 1024:.1f} MB (float32)')
print()
print(model)

Parameters: 803,922
Model size: ~3.1 MB (float32)

NeuralODEAutoencoder(
  (encoder): BiGRUEncoder(
    (gru): GRU(49, 128, num_layers=2, batch_first=True, bidirectional=True)
    (projection): Linear(in_features=256, out_features=32, bias=True)
  )
  (ode_func): ODEFunc(
    (net): Sequential(
      (0): Linear(in_features=33, out_features=128, bias=True)
      (1): SiLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): SiLU()
      (4): Linear(in_features=128, out_features=32, bias=True)
    )
  )
  (decoder): MLPDecoder(
    (net): Sequential(
      (0): Linear(in_features=32, out_features=128, bias=True)
      (1): SiLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): SiLU()
      (4): Linear(in_features=128, out_features=2450, bias=True)
    )
  )
)


## 2. Training Loop

In [5]:
# Hyperparameters
train_cfg = config['training']
EPOCHS = train_cfg['epochs']
LR = train_cfg['learning_rate']
WEIGHT_DECAY = train_cfg['weight_decay']
PATIENCE = train_cfg['early_stopping_patience']

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Epochs:          {EPOCHS}')
print(f'Learning rate:   {LR}')
print(f'Weight decay:    {WEIGHT_DECAY}')
print(f'Batch size:      {batch_size}')
print(f'Early stopping:  patience={PATIENCE}')
print(f'Train batches:   {len(loaders["train"])}')
print(f'Val batches:     {len(loaders["val"])}')

Epochs:          50
Learning rate:   0.001
Weight decay:    1e-05
Batch size:      256
Early stopping:  patience=5
Train batches:   273
Val batches:     75


In [6]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for X_batch, _ in loader:
        X_batch = X_batch.to(device)
        optimizer.zero_grad()
        x_hat, _ = model(X_batch)
        loss = nn.functional.mse_loss(x_hat, X_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


@torch.no_grad()
def validate(model, loader, device):
    """Validation loss on benign windows only."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for X_batch, y_batch in loader:
        benign_mask = y_batch == 0
        if benign_mask.sum() == 0:
            continue
        X_benign = X_batch[benign_mask].to(device)
        x_hat, _ = model(X_benign)
        loss = nn.functional.mse_loss(x_hat, X_benign)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

In [7]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'lr': [],
    'epoch_time': [],
}

# Early stopping state
best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
best_state_dict = None

print(f'Starting training for up to {EPOCHS} epochs...\n')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, loaders['train'], optimizer, device)
    val_loss = validate(model, loaders['val'], device)
    lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    elapsed = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(lr)
    history['epoch_time'].append(elapsed)

    # Early stopping
    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    marker = ' *' if improved else f' (patience {patience_counter}/{PATIENCE})'
    print(
        f'Epoch {epoch:3d}/{EPOCHS}  '
        f'train={train_loss:.6f}  val={val_loss:.6f}  '
        f'lr={lr:.2e}  time={elapsed:.1f}s{marker}'
    )

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val_loss:.6f} at epoch {best_epoch}')
print(f'Total training time: {sum(history["epoch_time"]):.0f}s ({sum(history["epoch_time"])/60:.1f} min)')

Starting training for up to 50 epochs...

Epoch   1/50  train=4430677735964432.000000  val=6307954668105018.000000  lr=1.00e-03  time=87.6s *


KeyboardInterrupt: 

## 3. Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Loss curves ---
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'o-', label='Train', markersize=4)
ax.plot(epochs_range, history['val_loss'], 's-', label='Val (benign)', markersize=4)
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Log-scale loss ---
ax = axes[1]
ax.plot(epochs_range, history['train_loss'], 'o-', label='Train', markersize=4)
ax.plot(epochs_range, history['val_loss'], 's-', label='Val (benign)', markersize=4)
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (log scale)')
ax.set_title('Loss (Log Scale)')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Learning rate ---
ax = axes[2]
ax.plot(epochs_range, history['lr'], 'g-o', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine Annealing LR Schedule')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/training_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Saved to figures/training_curves.png')

In [ ]:
# Epoch time analysis
times = history['epoch_time']
print(f'Epoch time:  mean={np.mean(times):.1f}s  min={np.min(times):.1f}s  max={np.max(times):.1f}s')
print(f'Total time:  {sum(times):.0f}s ({sum(times)/60:.1f} min)')

## 4. Save Best Checkpoint

In [ ]:
import os

checkpoint_dir = '../checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': best_state_dict,
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': history['train_loss'][best_epoch - 1],
    'val_loss': best_val_loss,
    'config': config,
    'history': history,
}

path = os.path.join(checkpoint_dir, 'best.pt')
torch.save(checkpoint, path)
print(f'Checkpoint saved to {path}')
print(f'  Epoch:     {best_epoch}')
print(f'  Val loss:  {best_val_loss:.6f}')
print(f'  Size:      {os.path.getsize(path) / 1024 / 1024:.1f} MB')

## 5. Quick Evaluation Preview

A quick look at anomaly score separation between benign and attack windows on the validation set. Full evaluation is in `05_evaluation.ipynb`.

In [ ]:
# Load best model
model.load_state_dict(best_state_dict)
model.eval()

# Compute anomaly scores on validation set
all_scores = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in loaders['val']:
        X_batch = X_batch.to(device)
        scores = model.anomaly_score(X_batch)
        all_scores.append(scores.cpu().numpy())
        all_labels.append(y_batch.numpy())

scores = np.concatenate(all_scores)
labels = np.concatenate(all_labels)

benign_scores = scores[labels == 0]
attack_scores = scores[labels == 1]

print(f'Benign scores:  mean={benign_scores.mean():.4f}  median={np.median(benign_scores):.4f}  std={benign_scores.std():.4f}')
print(f'Attack scores:  mean={attack_scores.mean():.4f}  median={np.median(attack_scores):.4f}  std={attack_scores.std():.4f}')
print(f'Separation:     attack_mean / benign_mean = {attack_scores.mean() / max(benign_scores.mean(), 1e-10):.2f}x')

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

auroc = roc_auc_score(labels, scores)
auprc = average_precision_score(labels, scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
ax = axes[0]
ax.hist(benign_scores, bins=100, alpha=0.6, label=f'Benign (n={len(benign_scores):,})', density=True)
ax.hist(attack_scores, bins=100, alpha=0.6, label=f'Attack (n={len(attack_scores):,})', density=True)
ax.set_xlabel('Anomaly Score (MSE)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Box plot
ax = axes[1]
bp = ax.boxplot(
    [benign_scores, attack_scores],
    labels=['Benign', 'Attack'],
    patch_artist=True,
    showfliers=False,
)
bp['boxes'][0].set_facecolor('#4ECDC4')
bp['boxes'][1].set_facecolor('#FF6B6B')
ax.set_ylabel('Anomaly Score (MSE)')
ax.set_title(f'Score Comparison  |  AUROC={auroc:.4f}  AUPRC={auprc:.4f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/training_eval_preview.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nAUROC: {auroc:.4f}')
print(f'AUPRC: {auprc:.4f}')

## Summary

| Metric | Value |
|--------|-------|
| Best epoch | see above |
| Best val loss | see above |
| Val AUROC | see above |
| Val AUPRC | see above |
| Total training time | see above |

The checkpoint is saved at `checkpoints/best.pt`. Proceed to full evaluation with `05_evaluation.ipynb`.